# 01 — Data Exploration

## 1. Introduction and objectives

Explore Olist e-commerce events as context for CallVerse BO1 customer interaction, BO2 service resolution, BO3 post-purchase operations, and BO4 data-driven decisions. Olist is **not observed contact-center data**: it does not establish contact time/reason, agent handling, staffing, waits, abandonment, or SLA. Late delivery can motivate a synthetic inquiry scenario but does not prove a contact occurred. Generated contacts and contact probabilities are synthetic or assumptions. This notebook does not train models, forecast contacts, or build a simulation.

## 2. Import libraries and configure paths

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR, "exists:", DATA_DIR.exists())
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory missing: {DATA_DIR}. Put Olist CSVs in data/raw/olist/.")

Project root: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse
Data directory: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse\data\raw\olist exists: True


## 3. Dataset inventory

In [2]:
EXPECTED_FILES = {"olist_customers_dataset.csv","olist_geolocation_dataset.csv","olist_order_items_dataset.csv",
"olist_order_payments_dataset.csv","olist_order_reviews_dataset.csv","olist_orders_dataset.csv",
"olist_products_dataset.csv","olist_sellers_dataset.csv","product_category_name_translation.csv"}
csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files: raise FileNotFoundError(f"No CSV files in {DATA_DIR}")
found = {f.name for f in csv_files}
print("Files:",len(csv_files),"Missing:",sorted(EXPECTED_FILES-found),"Unexpected:",sorted(found-EXPECTED_FILES))
display(pd.DataFrame([{"file":f.name,"size_mb":round(f.stat().st_size/1024**2,2),
"expected":f.name in EXPECTED_FILES} for f in csv_files]))

Files: 9 Missing: [] Unexpected: []


,file,size_mb,expected
0,olist_customers_dataset.csv,8.62,True
1,olist_geolocation_dataset.csv,58.44,True
2,olist_order_items_dataset.csv,14.72,True
3,olist_order_payments_dataset.csv,5.51,True
4,olist_order_reviews_dataset.csv,13.78,True
5,olist_orders_dataset.csv,16.84,True
6,olist_products_dataset.csv,2.27,True
7,olist_sellers_dataset.csv,0.17,True
8,product_category_name_translation.csv,0.00,True


## 4. Structural validation
Inspect samples and inferred types, then read each table once. Fail with the source filename if reading fails.

In [3]:
for f in csv_files:
    try: sample=pd.read_csv(f,nrows=5)
    except Exception as e: raise RuntimeError(f"Cannot inspect {f.name}: {e}") from e
    print("\n",f.name,list(sample.columns)); print(sample.dtypes)
datasets={}
for f in csv_files:
    try: datasets[f.stem]=pd.read_csv(f)
    except Exception as e: raise RuntimeError(f"Cannot load {f.name}: {e}") from e
display(pd.DataFrame([{"dataset":n,"rows":len(d),"columns":len(d.columns),
"memory_mb":round(d.memory_usage(index=True,deep=True).sum()/1024**2,2)} for n,d in datasets.items()]))
orders_df=datasets.get("olist_orders_dataset"); reviews_df=datasets.get("olist_order_reviews_dataset")


 olist_customers_dataset.csv ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

 olist_geolocation_dataset.csv ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

 olist_order_items_dataset.csv ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
frei

,dataset,rows,columns,memory_mb
0,olist_customers_dataset,99441,5,26.59
1,olist_geolocation_dataset,1000163,5,129.38
2,olist_order_items_dataset,112650,7,35.99
3,olist_order_payments_dataset,103886,5,16.23
4,olist_order_reviews_dataset,99224,7,39.12
5,olist_orders_dataset,99441,8,52.94
6,olist_products_dataset,32951,9,6.30
7,olist_sellers_dataset,3095,4,0.59
8,product_category_name_translation,71,2,0.01


## 5. Data dictionary and meaning
Roles below describe intended table meaning; actual columns are listed. Delivery delay does not prove contact. Reviews are not contact-center surveys.

In [ ]:
roles={"olist_customers_dataset":"Customer IDs/location; no contact logs.",
"olist_geolocation_dataset":"ZIP-prefix coordinates; prefix can repeat.",
"olist_order_items_dataset":"Order lines/product/seller/price/freight; one-to-many order.",
"olist_order_payments_dataset":"Payment type, installments and value; multiple per order possible.",
"olist_order_reviews_dataset":"Review scores/comments/times; not contact transcripts.",
"olist_orders_dataset":"Order status and purchase/approval/carrier/actual/estimated delivery dates.",
"olist_products_dataset":"Product category and attributes.","olist_sellers_dataset":"Seller IDs/location, not agents.",
"product_category_name_translation":"Category name mapping."}
display(pd.DataFrame([{"table":n,"role":roles.get(n,"Unexpected CSV; inspect."),
"columns_found":", ".join(d.columns.astype(str))} for n,d in datasets.items()]))

## 6. Data quality analysis

### Missing values, duplicates, identifiers
Missing delivery times may be expected for non-delivered orders. Report without repair. Repeated identifiers may be legitimate in child tables.

In [ ]:
miss=[]
for n,d in datasets.items():
    m=d.isna().sum(); miss.append({"table":n,"missing_cells":int(m.sum()),"columns_missing":int((m>0).sum())})
    print("\nMissingness:",n)
    display(pd.DataFrame({"column":m.index,"missing":m.values,"percent":m.values/len(d)*100 if len(d) else 0}).sort_values("percent",ascending=False).head(10))
display(pd.DataFrame(miss))
display(pd.DataFrame([{"table":n,"duplicate_rows":int(d.duplicated().sum())} for n,d in datasets.items()]))
keys={"olist_orders_dataset":["order_id"],"olist_customers_dataset":["customer_id","customer_unique_id"],
"olist_order_items_dataset":["order_id","order_item_id"],"olist_order_payments_dataset":["order_id"],
"olist_order_reviews_dataset":["review_id","order_id"],"olist_products_dataset":["product_id"],"olist_sellers_dataset":["seller_id"]}
r=[]
for n,cols in keys.items():
    d=datasets.get(n)
    if d is not None:
        for c in cols:
            if c in d:
                v=d[c].dropna(); r.append({"table":n,"key":c,"distinct":v.nunique(),
                "repeated_rows":int(v.duplicated(keep=False).sum()),"unique_ratio":v.nunique()/len(v) if len(v) else None})
display(pd.DataFrame(r))

### Timestamp parsing; numeric and categorical validation
Parse date-like fields separately and report conversion failures. Inspect value ranges/distributions; suspicious values are not deleted.

In [ ]:
parsed=[]
for n in ["olist_orders_dataset","olist_order_reviews_dataset"]:
    d=datasets.get(n)
    if d is not None:
        for c in [x for x in d if "date" in x.lower() or "timestamp" in x.lower()]:
            v=pd.to_datetime(d[c],errors="coerce")
            parsed.append({"table":n,"column":c,"invalid":int((d[c].notna()&v.isna()).sum())})
            d[c+"_parsed"]=v
display(pd.DataFrame(parsed))
numeric=[]; categorical=[]
for n,d in datasets.items():
    for c in d:
        if pd.api.types.is_numeric_dtype(d[c]):
            v=d[c].dropna(); numeric.append({"table":n,"column":c,"min":v.min() if len(v) else None,
            "median":v.median() if len(v) else None,"max":v.max() if len(v) else None,
            "negative_count":int((v<0).sum()) if len(v) else 0})
        elif d[c].nunique(dropna=True)<=30:
            categorical.append({"table":n,"column":c,"top_values":d[c].value_counts(dropna=False).head(8).to_dict()})
display(pd.DataFrame(numeric)); display(pd.DataFrame(categorical))

## 7. Relationship and referential integrity analysis
Key-set comparisons avoid large joins. Order keys may repeat in items/payments; reviews and delivery timestamps are not mandatory. Geolocation ZIP prefixes may repeat.

In [ ]:
spec=[("Customers→Orders","olist_orders_dataset","customer_id","olist_customers_dataset","customer_id"),
("Orders→Items","olist_order_items_dataset","order_id","olist_orders_dataset","order_id"),
("Orders→Payments","olist_order_payments_dataset","order_id","olist_orders_dataset","order_id"),
("Orders→Reviews","olist_order_reviews_dataset","order_id","olist_orders_dataset","order_id"),
("Items→Products","olist_order_items_dataset","product_id","olist_products_dataset","product_id"),
("Items→Sellers","olist_order_items_dataset","seller_id","olist_sellers_dataset","seller_id"),
("Products→Translation","olist_products_dataset","product_category_name","product_category_name_translation","product_category_name")]
out=[]
for label,cn,ck,pn,pk in spec:
    c,p=datasets.get(cn),datasets.get(pn)
    if c is None or p is None or ck not in c or pk not in p: out.append({"relationship":label,"status":"table/key missing"}); continue
    parent=set(p[pk].dropna().unique()); vals=c[ck]; v=vals.dropna(); bad=v.map(lambda x:x not in parent)
    out.append({"relationship":label,"child_rows":len(c),"distinct_keys":v.nunique(),"missing_keys":int(vals.isna().sum()),
    "unmatched_rows":int(bad.sum()),"unmatched_pct":float(bad.mean()*100) if len(bad) else 0})
display(pd.DataFrame(out))
g=datasets.get("olist_geolocation_dataset")
if g is not None and "geolocation_zip_code_prefix" in g:
    z=g.groupby("geolocation_zip_code_prefix",dropna=False).size()
    display(pd.DataFrame([{"rows":len(g),"distinct_prefix":g.geolocation_zip_code_prefix.nunique(),
    "repeated_prefixes":int((z>1).sum()),"max_rows_per_prefix":int(z.max())}]))

## 8. Order status and timestamp analysis

In [ ]:
if orders_df is None: print("Orders unavailable.")
else:
    if "order_status" in orders_df:
        s=orders_df.order_status.value_counts(dropna=False)
        display(pd.DataFrame({"orders":s,"percent":s/len(orders_df)*100}).sort_values("orders",ascending=False))
        s.plot(kind="bar",figsize=(8,4),title="Order status distribution")
        plt.xlabel("Status"); plt.ylabel("Orders"); plt.xticks(rotation=35); plt.tight_layout(); plt.show()
    p="order_purchase_timestamp_parsed"
    if p in orders_df:
        t=orders_df[p].dropna()
        print("Earliest:",t.min() if len(t) else None,"Latest:",t.max() if len(t) else None,
        "Span:",t.max()-t.min() if len(t) else None,"Excluded:",int(orders_df[p].isna().sum()))
        if len(t):
            m=t.dt.to_period("M").value_counts().sort_index(); m.index=m.index.to_timestamp()
            m.plot(figsize=(9,4),marker="o",title="Monthly order volume")
            plt.xlabel("Purchase month"); plt.ylabel("Orders"); plt.tight_layout(); plt.show()
    checks=[]
    for a,b,label in [("order_purchase_timestamp_parsed","order_approved_at_parsed","purchase before approval"),
    ("order_purchase_timestamp_parsed","order_delivered_carrier_date_parsed","purchase before carrier"),
    ("order_delivered_carrier_date_parsed","order_delivered_customer_date_parsed","carrier before delivery"),
    ("order_purchase_timestamp_parsed","order_delivered_customer_date_parsed","purchase before delivery")]:
        if a in orders_df and b in orders_df:
            both=orders_df[a].notna()&orders_df[b].notna()
            checks.append({"check":label,"comparable":int(both.sum()),"violations":int((both&(orders_df[a]>orders_df[b])).sum())})
    display(pd.DataFrame(checks))

## 9. Delivery performance analysis
Delay is actual minus estimated delivery in days; positive means late. Analyze delivered orders. Trend cohorts use purchase month, not delivery month. These events suggest synthetic scenarios, not actual inquiries.

In [ ]:
if orders_df is None: print("Orders unavailable.")
else:
    f={"purchase":"order_purchase_timestamp_parsed","estimate":"order_estimated_delivery_date_parsed",
    "actual":"order_delivered_customer_date_parsed","carrier":"order_delivered_carrier_date_parsed"}
    display(pd.DataFrame([{"field":k,"present":int(orders_df[c].notna().sum()),"percent":orders_df[c].notna().mean()*100}
    for k,c in f.items() if c in orders_df]))
    e,a=f["estimate"],f["actual"]
    if e in orders_df and a in orders_df:
        eligible=orders_df[e].notna()&orders_df[a].notna()
        orders_df["delivery_delay_days"]=(orders_df[a]-orders_df[e]).dt.total_seconds()/86400
        delivered=orders_df.order_status.eq("delivered") if "order_status" in orders_df else pd.Series(True,index=orders_df.index)
        d=orders_df.loc[eligible&delivered,"delivery_delay_days"]
        print("Eligible:",int(eligible.sum()),"Excluded:",int((~eligible).sum()),"Delivered analyzed:",len(d))
        if len(d):
            display(pd.DataFrame([{"mean":d.mean(),"median":d.median(),"late_pct":(d>0).mean()*100,
            "on_early_pct":(d<=0).mean()*100,"min":d.min(),"max":d.max()}]))
            d.plot(kind="hist",bins=40,figsize=(8,4),title="Delivery delay distribution")
            plt.xlabel("Actual - estimated days"); plt.ylabel("Orders"); plt.tight_layout(); plt.show()
            p=f["purchase"]
            if p in orders_df:
                q=orders_df.loc[eligible&delivered,[p,"delivery_delay_days"]].copy()
                q["purchase_month"]=q[p].dt.to_period("M").dt.to_timestamp()
                q.groupby("purchase_month").delivery_delay_days.median().plot(figsize=(9,4),marker="o",title="Median delay by purchase-month cohort")
                plt.xlabel("Purchase month"); plt.ylabel("Median days"); plt.tight_layout(); plt.show()
    else: print("Delay unavailable: timestamp column missing.")

## 10. Customer reviews and satisfaction signals
Review scores are not contact-center surveys; association does not prove causation.

In [ ]:
if reviews_df is None or "review_score" not in reviews_df: print("Reviews or score column unavailable.")
else:
    print("Missing scores:",int(reviews_df.review_score.isna().sum()))
    s=reviews_df.review_score.value_counts(dropna=False).sort_index()
    display(pd.DataFrame({"reviews":s,"percent":s/len(reviews_df)*100}))
    s.plot(kind="bar",title="Review score distribution"); plt.xlabel("Score"); plt.ylabel("Review rows"); plt.tight_layout(); plt.show()
    if orders_df is not None and "delivery_delay_days" in orders_df and "order_id" in orders_df and "order_id" in reviews_df:
        o=orders_df[["order_id","delivery_delay_days"]].drop_duplicates("order_id")
        j=reviews_df[["order_id","review_score"]].drop_duplicates().merge(o,on="order_id",how="left",validate="many_to_one")
        x=j.dropna(subset=["review_score","delivery_delay_days"])
        print("Review rows with delay:",len(x))
        if len(x): display(x.groupby("review_score").delivery_delay_days.agg(["count","median","mean"]))

## 11. Preliminary e-commerce insights
Summarize actual calculated observations; no statistical significance is claimed.

In [ ]:
rows=[]
if orders_df is not None:
    rows.append({"area":"Orders","finding":f"{len(orders_df):,} source order rows","type":"source"})
    if "order_status" in orders_df and len(orders_df):
        s=orders_df.order_status.value_counts().head(1)
        rows.append({"area":"Most common status","finding":f"{s.index[0]} ({int(s.iloc[0]):,})","type":"source"})
    if "delivery_delay_days" in orders_df:
        x=orders_df.delivery_delay_days.dropna()
        rows.append({"area":"Delay","finding":f"{len(x)} derived values; median {x.median():.2f}" if len(x) else "No valid values","type":"derived"})
rows.append({"area":"Support","finding":"No contact arrivals/reasons, handling times, or staffing recorded.","type":"limitation"})
display(pd.DataFrame(rows))

## 12. CallVerse relevance assessment

**BO1:** customer/product/order/category context informs scenarios; actual contacts/assignments are absent. **BO2:** order/payment/delivery/review context can ground synthetic inquiries; agent responses, resolutions, escalations, and handle time are absent. **BO3:** statuses, cancellations, dates, values inform post-purchase scenarios; schedules, costs, AHT are absent. **BO4:** supports order/delivery exploration and possibly order forecasting; order counts are not contact counts.

Provenance: source data is directly recorded; derived data is computed; synthetic data is generated; assumptions are rules requiring justification. Derived is not synthetic.

In [ ]:
display(pd.DataFrame([
{"category":"Source data","meaning":"Directly recorded","example":"Order status"},
{"category":"Derived data","meaning":"Computed","example":"Delivery delay"},
{"category":"Synthetic data","meaning":"Generated","example":"Simulated contact time"},
{"category":"Assumption","meaning":"Requires justification/calibration","example":"Contact probability for delay"}]))

## 13. Preliminary go/no-go decision
Olist is an e-commerce event substrate, not contact-center ground truth.

In [ ]:
display(pd.DataFrame([
{"use":"E-commerce exploration","decision":"Suitable","reason":"Linked commerce tables."},
{"use":"Order/delivery analytics","decision":"Suitable with limitations","reason":"Milestones exist; profile data quality."},
{"use":"Order-volume forecasting","decision":"Suitable with limitations","reason":"Order counts possible; not contact counts."},
{"use":"Synthetic scenario context","decision":"Suitable with limitations","reason":"Exception conditions; propensity assumed."},
{"use":"Actual contact/AHT training","decision":"Not supported alone","reason":"No support logs or handle durations."},
{"use":"Wait/abandonment/SLA validation","decision":"Not supported alone","reason":"No queue or staffing observations."},
{"use":"Observed workforce optimizer","decision":"Not supported alone","reason":"No skills, schedules, availability, wages."}]))
print("Later stages need real operational traces or explicit assumptions and sensitivity analysis.")

## 14. Final summary and next steps
Review inventory, missingness, parse errors, identifiers, joins, coverage, delivery, and reviews before drawing conclusions. Olist provides e-commerce IDs/status/timestamps/products/payments/reviews; delivery delay is derived. Contact reasons/times, AHT, staffing, wait, abandonment, SLA, and true contact-center satisfaction cannot be learned directly. Keep source/derived/synthetic/assumed distinct and do not silently remove rows. Next planned step: dedicated contact/scenario analysis in `02_contact_analysis.ipynb`. No forecasting or simulation is implemented here.